# Simulacro de parcial 2C 2026

Este archivo contiene los ejercicios 2 y 3. Las consignas completas se encuentran en el enunciado principal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.special import logsumexp
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix

np.set_printoptions(precision=3, suppress=True)

## Ejercicio 2

In [ ]:
rng = np.random.default_rng(7)

medias = {
    'a': np.array([-2.0, 1.0]),
    'i': np.array([ 2.0,-1.0]),
}
sigma = 0.45

def generar_secuencias(orden, cantidad):
    secuencias = []
    for _ in range(cantidad):
        n = rng.integers(8, 13)
        primera = rng.normal(medias[orden[0]], sigma, size=(n, 2))
        segunda = rng.normal(medias[orden[1]], sigma, size=(n, 2))
        secuencias.append(np.vstack([primera, segunda]))
    return secuencias

Y_train = {
    'ai': ### <-- COMPLETAR
    'ia': ### <-- COMPLETAR
}
Y_test = {
    'ai': ### <-- COMPLETAR
    'ia': ### <-- COMPLETAR
}

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, palabra in zip(axes, ['ai', 'ia']):
    Y = Y_train[palabra][0]
    ax.plot(Y[:, 0], marker='o', label='$c_1$')
    ax.plot(Y[:, 1], marker='o', label='$c_2$')
    ax.set_title(f'/{palabra}/')
    ax.set_xlabel('Frame $t$')
    ax.grid(alpha=0.25)
axes[0].set_ylabel('Valor del coeficiente')
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
def segmentar_uniforme(T, n_estados):
    """Asigna cada frame a un estado mediante una segmentación uniforme fija."""
    if T < n_estados:
        raise ValueError('La secuencia tiene menos frames que estados.')

    cortes = np.linspace(0, T, n_estados + 1).astype(int)
    estados = np.zeros(T, dtype=int)
    
    for s in range(n_estados):
        estados[cortes[s]:cortes[s+1]] = s
    # Si por redondeo quedó algún estado vacío, no afecta para T >> n_estados
    return estados

def log_gauss_diag(y, mu, var):
    """Log densidad de una gaussiana diagonal para cada fila de y."""
    D = y.shape[1]
    return -0.5 * (D * np.log(2 * np.pi) + np.sum(np.log(var)) + np.sum((y - mu) ** 2 / var, axis=1))


def estimar_hmm_uniforme(secuencia, n_estados=5, var_floor=1e-2):
    """Entrena un HMM left-to-right con una gaussiana diagonal por estado."""
    D = secuencia[0].shape[1]
    frames_por_estado = [[] for _ in range(n_estados)]
    transiciones = np.ones((n_estados, n_estados)) * 1e-3  # suavizado mínimo

    for y in secuencia:
        estados = segmentar_uniforme(len(y), n_estados)
        for s in range(n_estados):
            frames_por_estado[s].append(y[estados == s])
        for t in range(len(estados) - 1):
            i, j = estados[t], estados[t+1]
            if j == i or j == i + 1:
                transiciones[i, j] += 1

    # Forzamos topología left-to-right: self-loop y avance
    A = np.zeros((n_estados, n_estados))
    for i in range(n_estados - 1):
        A[i, i] = transiciones[i, i]
        A[i, i+1] = transiciones[i, i+1]
        A[i] /= A[i].sum()
    A[-1, -1] = 1.0

    mus = np.zeros((n_estados, D))
    vars_ = np.zeros((n_estados, D))
    for s in range(n_estados):
        ys = np.vstack(frames_por_estado[s])
        mus[s] = ys.mean(axis=0)
        vars_[s] = np.maximum(ys.var(axis=0), var_floor)

    pi = np.zeros(n_estados)
    pi[0] = 1.0
    return {'pi': pi, 'A': A, 'mu': mus, 'var': vars_}


def log_emisiones(y, modelo):
    return np.column_stack([log_gauss_diag(y, modelo['mu'][s], modelo['var'][s]) for s in range(modelo['mu'].shape[0])])


def log_probabilidades(P):
    """Convierte probabilidades a log, manteniendo los ceros como -inf."""
    P = np.asarray(P, dtype=float)
    logP = np.full(P.shape, -np.inf, dtype=float)

    mask = P > 0
    logP[mask] = np.log(P[mask])

    return logP


def log_forward(logB, pi, A):
    T, N = logB.shape
    log_alpha = np.full((T, N), -np.inf)
    log_pi = log_probabilidades(pi)
    log_A = log_probabilidades(A)

    log_alpha[0] = log_pi + logB[0]

    for t in range(1, T):
        for j in range(N):
            log_alpha[t, j] = logsumexp(log_alpha[t - 1] + log_A[:, j]) + logB[t, j]

    # Sumamos todos los caminos legales que terminan en el último estado
    score = log_alpha[-1, -1]

    return score, log_alpha

In [ ]:
modelos = {} ### <-- COMPLETAR: entrenar un modelo por pseudopalabra

def score_secuencia(Y, modelo):
    logB = ### <-- COMPLETAR
    score, _ = ### <-- COMPLETAR
    return score

def predecir_secuencia(Y, modelos):
    scores = ### <-- COMPLETAR
    return max(scores, key=scores.get)

verdaderas = []
predichas = []
for verdadera, secuencias in Y_test.items():
    for Y in secuencias:
        verdaderas.append(verdadera)
        predichas. ### <-- COMPLETAR

In [ ]:
acc = ### <-- COMPLETAR
cm = ### <-- COMPLETAR
print(f'Accuracy HMM: {acc:.3f}')
ConfusionMatrixDisplay(cm, display_labels=['/ai/', '/ia/']).plot(cmap='Blues')
plt.title('Matriz de confusión')
plt.show()

In [ ]:
media_ai = ### <-- COMPLETAR
media_ia = ### <-- COMPLETAR
distancia = ### <-- COMPLETAR
print('Media /ai/:', media_ai)
print('Media /ia/:', media_ia)
print('Distancia entre medias:', distancia)

### Interpretación del ejercicio 2

<mark>Completar<mark>


## Ejercicio 3

In [ ]:
fs = 8000
F0 = 120
duracion = 0.2
N_FFT = 4096
epsilon = 1e-12

n = np.arange(int(fs * duracion))
armonicos = np.arange(3, 11)
x = ### <-- COMPLETAR
xw = x * np.hanning(len(x))

In [ ]:
# El espectro unilateral se calcula con el mismo patron usado en P1.
X = np.fft.rfft(xw, n=N_FFT)
magnitud = np.abs(X)

cepstrum = ### <-- COMPLETAR 
frecuencias = np.fft.rfftfreq(N_FFT, d=1/fs)
quefrencias = np.arange(N_FFT) / fs

In [ ]:
mascara_pitch = ### <-- COMPLETAR 
q0 = ### <-- COMPLETAR 
F0_estimado = ### <-- COMPLETAR 
print(f'Quefrencia del pico: {1000*q0:.3f} ms')
print(f'F0 estimado: {F0_estimado:.2f} Hz')

fig, axes = plt.subplots(2, 1, figsize=(9, 6))
axes[0]. ### <-- COMPLETAR 
axes[0].set_xlim(0, 1500)
axes[0].set_xlabel('Frecuencia $f$ (Hz)')
axes[0].set_ylabel('$|X[k]|$')
axes[0].set_title('Espectro')
axes[0].grid(alpha=0.25)

axes[1]. ### <-- COMPLETAR 
axes[1].axvline(1000 * q0, color='tab:red', ls='--', label=f'{F0_estimado:.1f} Hz')
axes[1].set_xlim(2, 15)
axes[1].set_xlabel('Quefrencia $q_n$ (...)') ### <-- COMPLETAR unidades
axes[1].set_ylabel('$c[n]$')
axes[1].set_title('Cepstrum real')
axes[1].grid(alpha=0.25)
axes[1].legend()
plt.tight_layout()
plt.show()

### Interpretación del ejercicio 3

<mark>Completar<mark>